# Given a scentence
# A cat sat on the mat besides the door.



In [1]:
import torch

In [2]:
embeddings = torch.rand([5,6])

In [3]:
import torch.nn.functional as F
cosine_similarity = torch.zeros(embeddings.shape[0], embeddings.shape[0])
cosine_similarity
for i in range(embeddings.shape[0]):
    vector = embeddings[i]
    for j in range(embeddings.shape[0]):
        vector_y = embeddings[j]
        cosine_similarity[i,j] = F.cosine_similarity(vector, vector_y, dim = 0)
cosine_similarity

tensor([[1.0000, 0.6619, 0.8845, 0.5861, 0.9034],
        [0.6619, 1.0000, 0.8004, 0.8548, 0.6903],
        [0.8845, 0.8004, 1.0000, 0.8508, 0.8379],
        [0.5861, 0.8548, 0.8508, 1.0000, 0.5639],
        [0.9034, 0.6903, 0.8379, 0.5639, 1.0000]])

# Get attention scores.

In [4]:
#sum across full row.
display(cosine_similarity.sum(dim=1))
# this gives us a matrix of [5] values.
# now when we divide a matrix [5*5] ...it is divided by this [1*5] -> [5*5] by broadcasting.
# one row , divided by 5 different numbers, which is wrong.

tensor([4.0358, 4.0073, 4.3734, 3.8555, 3.9954])

In [5]:
attention = cosine_similarity.T / cosine_similarity.sum(dim=0)
attention = attention.T

In [6]:
attention.sum(dim = 1)

tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000])

In [7]:
after_attention = attention @ embeddings
after_attention

tensor([[0.3536, 0.3630, 0.5748, 0.3687, 0.7123, 0.4523],
        [0.4566, 0.4024, 0.4613, 0.3472, 0.7080, 0.4690],
        [0.4062, 0.3924, 0.5149, 0.3624, 0.6990, 0.4732],
        [0.4784, 0.4311, 0.4341, 0.3551, 0.6886, 0.4944],
        [0.3533, 0.3526, 0.5735, 0.3583, 0.7229, 0.4547]])

# this will be the vectorys with context.

# Now how to get these vectors in transformers.
## Use learnable weights to get Q, K, V matrix for the input embedding.

In [9]:
import torch
input_token_embedding = torch.rand(5,6)
display(input_token_embedding)
input_token_embedding.shape

tensor([[0.9454, 0.1548, 0.8046, 0.1607, 0.2714, 0.0874],
        [0.5279, 0.2141, 0.5391, 0.8094, 0.9220, 0.0425],
        [0.7490, 0.9357, 0.8951, 0.6405, 0.8132, 0.7672],
        [0.3992, 0.4457, 0.2549, 0.2363, 0.7112, 0.7584],
        [0.4147, 0.4581, 0.0415, 0.7808, 0.4650, 0.4493]])

torch.Size([5, 6])

# input is 5*6
# this is 5 tokens, with each having 6 dimension vector.
# this is your input vector size.
# what you want as output_vector size.

In [10]:
tokens = input_token_embedding.shape[0]
input_dim = input_token_embedding.shape[1]
output_dim = 3
print("input dim: ",input_dim,", and output is :", output_dim)

input dim:  6 , and output is : 3


#your Weight_W should be converting that input dim to output_dim.

In [11]:
Q_weights = torch.nn.Parameter(torch.rand(input_dim,output_dim), requires_grad=False)
K_weights = torch.nn.Parameter(torch.rand(input_dim,output_dim), requires_grad=False)
V_weights = torch.nn.Parameter(torch.rand(input_dim,output_dim), requires_grad=False)

In [12]:
token_Q = input_token_embedding @ Q_weights # this will convert the token embedding vectors to that of output_dim
token_K = input_token_embedding @ K_weights
token_V = input_token_embedding @ V_weights

In [13]:
token_Q.shape , token_K.shape, token_V.shape

(torch.Size([5, 3]), torch.Size([5, 3]), torch.Size([5, 3]))

In [17]:
#now use this to find attention weights.
# QKT to get the attention.
import math
QKT  = (token_Q @ token_K.T)/math.sqrt(output_dim)
# each row here has attention scores for Qwery_i with all the Keys.

In [26]:
# We need to normalize them, for them to be interpretable.
# get attention weights from QKT

# QKT = 5 * 5 => sum dim =1 => 1 * 5
# 5*5 should be divided by 1 * 5  => 

QKT_normalized = QKT.T/ QKT.sum(dim = 1)
QKT_normalized = QKT_normalized.T


QKT_normalized_softmax = torch.softmax(QKT, dim = 1)#.sum(dim=1)

In [27]:
# use attention weights to combine value vectors to get outputu.
# attention wiehgts -> to get back token embeddings of side output_dim
# for the first token, what is the token embedding how should i combine other Value wector to get the
# resultant output vector.
# how other value should be combined will be decided by Key and Value vectors.
output = QKT_normalized_softmax @ token_V
output.shape, output


(torch.Size([5, 3]),
 tensor([[2.7429, 1.8151, 1.8587],
         [3.0639, 2.0372, 2.0664],
         [3.2955, 2.2055, 2.2169],
         [2.9682, 1.9701, 2.0031],
         [2.9760, 1.9749, 2.0092]]))

In [ ]:
# the token embedding is now changed to ouput context vector
input_token_embedding.shape, output.shape

(torch.Size([5, 6]), torch.Size([5, 3]))

# this output context vector corresponding to each token is a semantically enriched vector.
# corresponding to each token.

# why do we divide by sqrt of dim

In [40]:
sample = torch.randn(1024)
print(max(torch.softmax(sample , dim = -1 )) , max(torch.softmax(sample/math.sqrt(1024), dim =-1)))
print(min(torch.softmax(sample , dim = -1 )) , min(torch.softmax(sample/math.sqrt(1024), dim =-1)))

#without division...some get very high value..others get 0 as its exponential
# with division,...kind of weights get distributed accross the token size.


tensor(0.0130) tensor(0.0011)
tensor(2.3439e-05) tensor(0.0009)
